# ML-04 · Data contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tessa-Saumu/FlyRank-ML-Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

> **Freestyle submission.** This notebook belongs to my final, freestyle capstone: *which ranking pages are losing clicks to AI-answer search before it shows, and which should an editor protect first?* My earlier Lane 2 (refresh) experiments are kept for the record in `work/archive/` and are not part of this submission.

**What this card asks:** five plain-word contract answers, exactly three verification queries on a mid-panel month, at
most five features each with "knowable at the decision moment because...", and a deliberate label-derived leak shown,
then removed. **The principle it serves:** data hygiene first. One explicit grain, null is not zero, and nothing from
the future in a feature.

## 1. The contract in five answers

1. **What is one row?** One top-10 page at one monthly decision point (page x origin month), built from the daily
   performance fact table (report_date x client x content).
2. **Which tables?** `fact_content_daily_performance` (Search Console and GA4, daily), `dim_content` (page
   metadata) and `dim_clients` (data start dates). Warehouse build v20260703, pinned revision `50cbf7c3`.
3. **Which window?** Features use the two closed months before the decision. The label uses the two months after it.
   Origins run December 2025 to May 2026, after Google's September 2025 change that removed deep-result impressions.
4. **Label or proxy?** Observed: clicks in the outcome window below 75% of the clicks the page's current CTR predicts at
   its new position.
5. **One exclusion.** Pages whose expected outcome clicks are under 10 (4% to 13% per origin, mostly pages that lost
   visibility) are excluded because their label cannot be measured. This uses outcome-window impressions, so it is a
   disclosed population choice.

**Privacy rule I added.** The column `dim_content.provider_used` contains strings that look like client brand names in
some rows. It is never selected anywhere in this project, and I flagged it to my mentor.

In [1]:
# Setup: works locally and in Colab. Installs only missing packages and never prints a token.
import importlib.util, json, os, sys
if any(importlib.util.find_spec(p) is None for p in ["duckdb", "huggingface_hub", "sklearn", "scipy", "matplotlib"]):
    get_ipython().run_line_magic("pip", "install -q duckdb huggingface_hub pandas scikit-learn scipy matplotlib")
here = os.path.abspath(os.getcwd())
ROOT = next((p for p in [here, os.path.dirname(here), os.path.dirname(os.path.dirname(here))]
             if os.path.isdir(os.path.join(p, "work", "scripts"))), None)
if ROOT is None:  # fresh Colab runtime: fetch the code (no data lives in git)
    get_ipython().system("git clone -q https://github.com/Tessa-Saumu/FlyRank-ML-Internship.git")
    ROOT = os.path.abspath("FlyRank-ML-Internship")
sys.path.insert(0, os.path.join(ROOT, "work", "scripts"))
OUT, FIG = os.path.join(ROOT, "work", "outputs"), os.path.join(ROOT, "work", "figures")
import numpy as np, pandas as pd
import warnings; warnings.filterwarnings("ignore")
from IPython.display import Image, display, Markdown
import wh
DATA = wh.data_available(try_download=True)   # needs HF_TOKEN as an env var or Colab secret
print("pinned warehouse snapshot available:", DATA, "| revision", wh.REV[:8])
receipt = lambda name: json.load(open(os.path.join(OUT, name)))
pd.set_option("display.width", 170); pd.set_option("display.max_columns", 30)

pinned warehouse snapshot available: True | revision 50cbf7c3


## 2. Three verification queries on a mid-panel month (2026-03)

**Why March 2026:** it sits mid-panel, away from the sealed final months. These three checks prove the grain, the size
and date span, and the availability handling before anything is built on them.

In [2]:
if DATA:
    c = wh.con()
    d = wh.DAILY_DIR + "/month=2026-03/*.parquet"
    q1 = c.sql(f'''SELECT report_date, content_hash_id, COUNT(*) n FROM read_parquet('{d}')
                   GROUP BY 1, 2 HAVING COUNT(*) > 1 LIMIT 5''').df()
    q2 = c.sql(f'''SELECT COUNT(*) rows_, MIN(report_date)::VARCHAR first_day, MAX(report_date)::VARCHAR last_day,
                   COUNT(DISTINCT client_hash_id) clients, COUNT(DISTINCT content_hash_id) pages FROM read_parquet('{d}')''').df()
    q3 = c.sql(f'''SELECT SUM((gsc_data_available IS TRUE)::INT) gsc_true, SUM((gsc_data_available IS NOT TRUE)::INT) gsc_not_true,
                   SUM((gsc_data_available IS NULL)::INT) gsc_null, SUM((ga4_data_available IS TRUE)::INT) ga4_true,
                   SUM((ga4_data_available IS NULL)::INT) ga4_null FROM read_parquet('{d}')''').df()
    contract = {"q1_duplicate_grain_rows": len(q1), "q2": q2.iloc[0].to_dict(), "q3": {k: int(v) for k, v in q3.iloc[0].items()}}
    json.dump(contract, open(os.path.join(OUT, "data_contract.json"), "w"), indent=1, default=str)
contract = receipt("data_contract.json")
print("Q1 grain probe, duplicate (report_date, content) rows:", contract["q1_duplicate_grain_rows"], "(must be 0)")
print("Q2 size and span:", contract["q2"])
print("Q3 availability flags (three-valued, so filter with IS TRUE):", contract["q3"])

Q1 grain probe, duplicate (report_date, content) rows: 0 (must be 0)
Q2 size and span: {'rows_': 9841378, 'first_day': '2026-03-01', 'last_day': '2026-03-31', 'clients': 55, 'pages': 331437}
Q3 availability flags (three-valued, so filter with IS TRUE): {'gsc_true': 3611061, 'gsc_not_true': 6230317, 'gsc_null': 0, 'ga4_true': 413966, 'ga4_null': 3018741}


**What it means.** The grain holds (zero duplicates), March has the expected one-month span, and the availability flags
contain NULLs as well as TRUE and FALSE. That is why every query in this project filters with `IS TRUE`, and why GA4
columns are summed only on days where `ga4_data_available IS TRUE` (earlier days are zero-filled, and zero there does
not mean "no engagement").

## 3. Five core features, each knowable at the decision moment

The full model uses 34 engineered features (ML-08). The contract fixes the five that carry the core idea:

| Feature | Meaning | Knowable at the decision moment because... |
|---|---|---|
| `l_imp_f` | log impressions over months *o-2* and *o-1* | both months close before day 1 of month *o* |
| `pos_f` | impression-weighted average position, same window | same window, computed from `gsc_sum_position / impressions` |
| `ctr_vs_ref` | log of shrunk CTR over the fixed position curve | the curve uses Oct-Nov 2025 only, before every outcome window |
| `ctr_growth` | log CTR change from month *o-2* to month *o-1* | both months are closed |
| `page_prev_ratio` | the page's own clicks-vs-expected two origins ago | that origin's outcome window ended at month *o-1* |

## 4. A deliberate leak, shown and then removed

**Why:** to prove the evaluation can see a leak. I add the outcome-window clicks as a "feature" (the label's own
ingredient), train on December to January, and score March.

In [3]:
if DATA:
    from erosion_model import load, make_features, lr_model, p_at_k, tiebreak
    from sklearn.metrics import average_precision_score
    df, grid, vals = load(); df = df[df.measurable.fillna(False)].reset_index(drop=True)
    X5 = make_features(df, grid, vals)[["l_imp_f", "pos_f", "ctr_vs_ref", "ctr_growth", "page_prev_ratio"]]
    y = df.erode.to_numpy().astype(int)
    tr, te = df.origin.isin(["2025-12", "2026-01"]).to_numpy(), (df.origin == "2026-03").to_numpy()
    tb = tiebreak(df.content_hash_id[te]); res = {}
    for name, X in [("five_features", X5), ("with_leak_clk_out", X5.assign(leak_clk_out=np.log1p(df.clk_out.fillna(0)) - np.log1p(df.exp_out)))]:
        cols = [c for c in X.columns if X.loc[tr, c].notna().any()]
        s = lr_model(0.1).fit(X.loc[tr, cols], y[tr]).predict_proba(X.loc[te, cols])[:, 1]
        res[name] = {"p@50": round(p_at_k(y[te], s + tb, 50), 3), "pr_auc": round(float(average_precision_score(y[te], s)), 3)}
    res["base_rate_2026-03"] = round(float(y[te].mean()), 3)
    contract = receipt("data_contract.json"); contract["leak_demo"] = res
    json.dump(contract, open(os.path.join(OUT, "data_contract.json"), "w"), indent=1, default=str)
display(pd.DataFrame(receipt("data_contract.json")["leak_demo"]).T)

,p@50,pr_auc
five_features,0.360,0.376
with_leak_clk_out,1.000,1.000
base_rate_2026-03,0.338,0.338


**What it means.** With the outcome's own ingredient as a feature, the ranking becomes perfect. That is the signature
of a leak, not skill. With it removed, these five features alone rank only slightly above the base rate (P@50 0.36
against 0.338). The core idea is real but weak on its own, which is why ML-08 engineers 34 features: feature
engineering, not model choice, is where this study earns its signal. The leaky column is never used again. Feature timing is asserted in code (`erosion_frame.py` builds features only from months
before the origin).

## 5. One named limitation

Search Console does not record whether an AI answer appeared for a query. So this study measures click loss at a stable
rank, which is **consistent with** AI answers absorbing clicks. Seasonality, changes to other search features and
impression inflation are competing explanations. ML-06 tests the ones the data allows.